# Cosmetique AI — runtime GPU reproductible

Ce notebook lance le pipeline V1 et son API privée sur un runtime Colab GPU. Il ne reçoit que l’image uploadée et une requête validée : aucune donnée PostgreSQL, clé de stockage ou identité utilisateur.

**Mode normal :** `Runtime > Run all`, autoriser Google Drive, puis copier l’URL temporaire, l’identifiant de runtime et le jeton dans l’environnement serveur du backend. **Mode démo directe :** activer `RUN_DIRECT_DEMO` dans le formulaire avant `Run all`.

Les poids sont liés à des commits Hugging Face immuables. Le produit source n’est jamais régénéré; seul le décor vide est génératif. Le tunnel Quick Tunnel est réservé à cette démonstration locale/session et n’est pas un hébergement GPU permanent.

In [ ]:
# @title 1. Monter Drive et installer le paquet versionné
from pathlib import Path
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive', force_remount=False)
matches = sorted(Path('/content/drive/MyDrive').rglob('Stage_1_ouvrier/ai_core/pyproject.toml'))
if len(matches) != 1:
    raise RuntimeError(f'Un seul Stage_1_ouvrier/ai_core est requis; trouvé: {len(matches)}')
AI_CORE_DIR = matches[0].parent
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', f'{AI_CORE_DIR}[service,gpu]'],
    check=True,
)
# Colab often leaves a broken Pillow mix; pin cleanly after install.
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '--quiet',
        '--force-reinstall',
        '--no-cache-dir',
        'Pillow==12.3.0',
    ],
    check=True,
)
print(f'Paquet installé depuis: {AI_CORE_DIR}')
print('Pillow 12.3.0 réinstallé proprement pour Colab.')
print('PyTorch Colab existant conservé volontairement; aucun remplacement automatique.')

In [ ]:
# @title 2. Préflight GPU, dépendances, modèles et police
import json

from ai_core.model_registry import pinned_model_refs, runtime_prerequisite_report

PREFLIGHT = runtime_prerequisite_report()
print(json.dumps(PREFLIGHT, indent=2, ensure_ascii=False))
print('\nRévisions modèles:')
for role, ref in pinned_model_refs().items():
    print(f'- {role}: {ref.repo_id}@{ref.revision} [{ref.license}]')
if not PREFLIGHT['ready']:
    raise RuntimeError('Préflight refusé: CUDA, versions épinglées ou police Manrope non conformes.')
if PREFLIGHT['gpu']['vram_bytes'] < 12 * 1024**3:
    raise RuntimeError('Au moins 12 Gio de VRAM sont requis pour le chargement séquentiel V1.')

In [ ]:
# @title 3. Mode et champs de la démo directe (optionnelle)
RUN_DIRECT_DEMO = False # @param {type:"boolean"}
PRODUCT_NAME = 'Sérum Éclat' # @param {type:"string"}
BRAND = 'Maison Exemple' # @param {type:"string"}
CATEGORY = 'Sérum' # @param {type:"string"}
LANGUAGE = 'fr' # @param ['fr', 'en']
AUDIENCE = '' # @param {type:"string"}
BENEFITS = 'Hydrate la peau' # @param {type:"string"}
INGREDIENTS = '' # @param {type:"string"}
VERIFIED_CLAIMS = '' # @param {type:"string"}
CTA = 'Découvrir' # @param {type:"string"}
CREATIVE_DIRECTION = 'Studio perle, lumière douce, accent framboise profond' # @param {type:"string"}
SEED = 42 # @param {type:"integer", min:0, max:4294967295}

print('Séparer plusieurs faits par « ; ». Tous les faits doivent déjà être dans la langue choisie.')

In [ ]:
# @title 4. Construire le pipeline lazy et l’API authentifiée
import secrets
import uuid

import torch
from ai_core.adapters import (
    GroundingDinoDetector,
    QwenEvidenceSelector,
    SamSegmenter,
    SdxlBackgroundGenerator,
)
from ai_core.model_registry import pinned_model_refs, runtime_dependency_refs
from ai_core.orchestrator import PipelineOrchestrator
from ai_core.service import GPUHealth, RuntimeSettings, create_app

RUNTIME_ID = f'colab-{uuid.uuid4()}'
RUNTIME_TOKEN = secrets.token_urlsafe(48)
orchestrator = PipelineOrchestrator(
    detector=GroundingDinoDetector(),
    segmenter=SamSegmenter(),
    background_generator=SdxlBackgroundGenerator(),
    copy_generator=QwenEvidenceSelector(),
    runtime_id=RUNTIME_ID,
    models=pinned_model_refs(),
    dependencies=runtime_dependency_refs(),
)

def gpu_probe():
    available = bool(torch.cuda.is_available())
    return GPUHealth(
        available=available,
        name=torch.cuda.get_device_name(0) if available else None,
        vram_bytes=int(torch.cuda.get_device_properties(0).total_memory) if available else 0,
    )

app = create_app(
    RuntimeSettings(token=RUNTIME_TOKEN, runtime_id=RUNTIME_ID),
    orchestrator=orchestrator,
    gpu_probe=gpu_probe,
    readiness_probe=orchestrator.preflight,
)
print('Contrats chargés; aucun poids GPU n’a encore été chargé.')

In [ ]:
# @title 5. Démarrer le serveur local et vérifier sa santé
import threading
import time
import urllib.request

import uvicorn

if 'SERVER' in globals():
    SERVER.should_exit = True
    time.sleep(1)
SERVER = uvicorn.Server(
    uvicorn.Config(app, host='127.0.0.1', port=8000, log_level='warning', access_log=False)
)
SERVER_THREAD = threading.Thread(target=SERVER.run, daemon=True)
SERVER_THREAD.start()
deadline = time.time() + 30
while not SERVER.started and time.time() < deadline:
    time.sleep(0.1)
if not SERVER.started:
    raise RuntimeError('Le serveur FastAPI local n’a pas démarré.')
health_request = urllib.request.Request(
    'http://127.0.0.1:8000/v1/health',
    headers={'Authorization': f'Bearer {RUNTIME_TOKEN}'},
)
with urllib.request.urlopen(health_request, timeout=10) as response:
    LOCAL_HEALTH = json.loads(response.read())
if not LOCAL_HEALTH['ready']:
    raise RuntimeError(f'Runtime local non prêt: {LOCAL_HEALTH}')
print(f"API locale prête — runtime {LOCAL_HEALTH['runtime_id']} — GPU {LOCAL_HEALTH['gpu']['name']}")

In [ ]:
# @title 6. Ouvrir le Cloudflare Quick Tunnel temporaire
import hashlib
import os
import queue
import re
import subprocess
import urllib.request

CLOUDFLARED_VERSION = '2026.7.2'
CLOUDFLARED_SHA256 = 'ec905ea7b7e327ff8abdde8cb64697a2152de74dbcdbf6aec9db8364eb3886cd'
CLOUDFLARED_PATH = Path('/content/cloudflared')
CLOUDFLARED_URL = (
    f'https://github.com/cloudflare/cloudflared/releases/download/'
    f'{CLOUDFLARED_VERSION}/cloudflared-linux-amd64'
)
if not CLOUDFLARED_PATH.exists() or hashlib.sha256(CLOUDFLARED_PATH.read_bytes()).hexdigest() != CLOUDFLARED_SHA256:
    urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED_PATH)
if hashlib.sha256(CLOUDFLARED_PATH.read_bytes()).hexdigest() != CLOUDFLARED_SHA256:
    raise RuntimeError('Checksum cloudflared invalide.')
CLOUDFLARED_PATH.chmod(0o755)
version_text = subprocess.check_output([str(CLOUDFLARED_PATH), '--version'], text=True)
if CLOUDFLARED_VERSION not in version_text:
    raise RuntimeError(f'Version cloudflared inattendue: {version_text}')
if 'TUNNEL_PROCESS' in globals() and TUNNEL_PROCESS.poll() is None:
    TUNNEL_PROCESS.terminate()
TUNNEL_PROCESS = subprocess.Popen(
    [str(CLOUDFLARED_PATH), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
tunnel_lines = queue.Queue()
def collect_tunnel_output():
    for line in TUNNEL_PROCESS.stdout:
        tunnel_lines.put(line)
threading.Thread(target=collect_tunnel_output, daemon=True).start()
pattern = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
TUNNEL_URL = None
deadline = time.time() + 60
while time.time() < deadline and TUNNEL_URL is None:
    try:
        line = tunnel_lines.get(timeout=1)
    except queue.Empty:
        if TUNNEL_PROCESS.poll() is not None:
            break
        continue
    match = pattern.search(line)
    if match:
        TUNNEL_URL = match.group(0)
if TUNNEL_URL is None:
    TUNNEL_PROCESS.terminate()
    raise RuntimeError('Cloudflare Quick Tunnel indisponible; consulter les dernières lignes du processus.')
for attempt in range(20):
    try:
        public_request = urllib.request.Request(
            f'{TUNNEL_URL}/v1/health',
            headers={'Authorization': f'Bearer {RUNTIME_TOKEN}'},
        )
        with urllib.request.urlopen(public_request, timeout=15) as response:
            PUBLIC_HEALTH = json.loads(response.read())
        if PUBLIC_HEALTH['runtime_id'] == RUNTIME_ID and PUBLIC_HEALTH['ready']:
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Le tunnel existe mais le healthcheck authentifié a échoué.')
print('Tunnel authentifié vérifié.')

In [ ]:
# @title 7. Valeurs serveur — copier une fois, puis effacer les sorties
from IPython.display import Markdown, display

display(Markdown(
    '### Runtime prêt\n'
    f'- `AI_SERVICE_URL={TUNNEL_URL}`\n'
    f'- `AI_RUNTIME_ID={RUNTIME_ID}`\n'
    f'- `AI_SERVICE_TOKEN={RUNTIME_TOKEN}`\n\n'
    '**Secret :** ces valeurs vont uniquement dans l’environnement serveur. '
    'Effacez les sorties du notebook avant de l’enregistrer ou de le partager.'
))

In [ ]:
# @title 8. Démo directe optionnelle et ZIP validé
if RUN_DIRECT_DEMO:
    import io
    import time
    from google.colab import files
    import httpx
    from PIL import Image, ImageOps
    from ai_core.artifacts import validate_bundle

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError('Téléverser exactement une image produit autorisée.')
    filename, source = next(iter(uploaded.items()))
    with Image.open(io.BytesIO(source)) as probe:
        source_format = probe.format
        corrected = ImageOps.exif_transpose(probe)
        width, height = corrected.size
    mime = {'JPEG': 'image/jpeg', 'PNG': 'image/png', 'WEBP': 'image/webp'}.get(source_format)
    if mime is None:
        raise RuntimeError('Formats acceptés: JPEG, PNG ou WebP.')
    split_values = lambda value: [item.strip() for item in value.split(';') if item.strip()]
    snapshot = {
        'product': {
            'name': PRODUCT_NAME,
            'brand': BRAND or None,
            'category': CATEGORY,
            'original_mime': mime,
            'original_sha256': hashlib.sha256(source).hexdigest(),
            'original_width': width,
            'original_height': height,
        },
        'generation': {
            'language': LANGUAGE,
            'seed': SEED,
            'audience': AUDIENCE or None,
            'benefits': split_values(BENEFITS),
            'ingredients': split_values(INGREDIENTS),
            'verified_claims': split_values(VERIFIED_CLAIMS),
            'cta': CTA or None,
            'creative_direction': CREATIVE_DIRECTION or None,
            'target_hint': None,
            'source_generation_id': None,
        },
    }
    canonical = json.dumps(snapshot, ensure_ascii=False, sort_keys=True, separators=(',', ':')).encode()
    envelope = {
        'generation_id': f'demo-{uuid.uuid4()}',
        'request_id': f'request-{uuid.uuid4()}',
        'input_snapshot_hash': hashlib.sha256(canonical).hexdigest(),
        'input': snapshot,
    }
    headers = {'Authorization': f'Bearer {RUNTIME_TOKEN}'}
    with httpx.Client(base_url='http://127.0.0.1:8000', headers=headers, timeout=60) as client:
        created = client.post(
            '/v1/jobs',
            data={'request': json.dumps(envelope, ensure_ascii=False, separators=(',', ':'))},
            files={'image': (filename, source, mime)},
        )
        created.raise_for_status()
        job_id = created.json()['id']
        while True:
            state = client.get(f'/v1/jobs/{job_id}')
            state.raise_for_status()
            body = state.json()
            print(f"{body['status']} — {body.get('stage')}")
            if body['status'] == 'done':
                break
            if body['status'] == 'error':
                raise RuntimeError(json.dumps(body, ensure_ascii=False))
            time.sleep(2)
        downloaded = client.get(
            f'/v1/jobs/{job_id}/bundle',
            headers={'X-Expected-Runtime-ID': RUNTIME_ID},
        )
        downloaded.raise_for_status()
    validate_bundle(downloaded.content)
    ZIP_PATH = Path('/content/cosmetique-ai-campaign.zip')
    ZIP_PATH.write_bytes(downloaded.content)
    files.download(str(ZIP_PATH))
else:
    print('Mode service actif. La démo directe est désactivée; aucune image n’est demandée.')

## Gate externe restant

Le notebook est construit pour un T4 frais, mais sa validation d’acceptation doit être enregistrée sur un vrai runtime : succès de `Run all`, révisions affichées, génération froide/chaude chronométrée, ZIP validé, puis test du site Docker. Ne jamais remplacer cette preuve par un résultat simulé.